# APS360 — RNN Model Variant: ResumeMatchNet-RNN

This notebook trains and evaluates the **RNN variant** of ResumeMatchNet,
which replaces pre-computed SBERT embeddings with a trainable **Bidirectional LSTM**
encoder. Everything else (Branch 1, Word2Vec, merge head) stays identical,
allowing a controlled comparison.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from rnn_dataset import RNNResumeDataset, rnn_collate_fn
from rnn_model import ResumeMatchNetRNN
from train_rnn import _evaluate, _split_indices, MATCH_THRESHOLD
from torch.utils.data import DataLoader, Subset

## 1. Load the dataset

In [ ]:
dataset = RNNResumeDataset(
    csv_path='../data/cleaned_resume_data.csv',
    cache_dir='../data/cache',
    fit=False,
)
print(f'Samples: {len(dataset)}  branch1_dim={dataset.branch1_dim}  vocab={len(dataset.vocab)}')

## 2. Model architecture

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = ResumeMatchNetRNN(
    branch1_dim=dataset.branch1_dim,
    vocab_size=len(dataset.vocab),
    w2v_dim=dataset.w2v.shape[1],
    embed_dim=128,
    rnn_hidden=192,   # bidirectional → 384 (matches SBERT dim)
    rnn_layers=2,
    hidden_dim=512,
    dropout=0.3,
).to(device)

print(f'\nTotal trainable parameters: {model.count_parameters():,}')
print(f'  - TextRNN (shared BiLSTM): {sum(p.numel() for p in model.text_rnn.parameters()):,}')
print(f'  - Branch 1 FC:             {sum(p.numel() for p in model.branch1_fc.parameters()):,}')
print(f'  - Branch 2 FC:             {sum(p.numel() for p in model.branch2_fc.parameters()):,}')
print(f'  - Merge head:              {sum(p.numel() for p in model.merged_fc.parameters()):,}')

## 3. Train

Uses the same training recipe as the SBERT model:
- 60/20/20 split, seed=42 (identical indices)
- Adam (lr=1e-3, weight_decay=1e-5) + CosineAnnealingLR
- Early stopping (patience=10)

In [ ]:
from train_rnn import train

results = train(
    csv_path='../data/cleaned_resume_data.csv',
    cache_dir='../data/cache',
    output_dir='../data/primary_model',
    epochs=50,
    batch_size=64,
    lr=1e-3,
    hidden_dim=512,
    dropout=0.3,
    patience=10,
)

## 4. Results summary

In [ ]:
print('=== RNN Model — Test Results ===')
print(f"RMSE: {results['test_rmse']:.4f}")
print(f"MAE:  {results['test_mae']:.4f}")
print(f"R²:   {results['test_r2']:.4f}")
print(f"Acc:  {results['test_acc']:.4f}")
print(f"F1:   {results['test_f1']:.4f}")

## 5. Learning curves

In [ ]:
h = results['history']
epochs_run = len(h['train_rmse'])
x = range(1, epochs_run + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(x, h['train_rmse'], label='Train', color='steelblue')
ax.plot(x, h['val_rmse'],   label='Val',   color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE'); ax.set_title('RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x, h['train_r2'], label='Train', color='steelblue')
ax.plot(x, h['val_r2'],   label='Val',   color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('R²'); ax.set_title('R²')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(x, h['train_acc'], label='Train Acc', color='steelblue')
ax.plot(x, h['val_acc'],   label='Val Acc',   color='tomato')
ax.plot(x, h['train_f1'],  label='Train F1',  color='steelblue', ls='--')
ax.plot(x, h['val_f1'],    label='Val F1',    color='tomato',    ls='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score'); ax.set_title('Accuracy & F1')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('ResumeMatchNet-RNN — Learning Curves', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Compare vs SBERT model and Baseline

In [ ]:
# Load SBERT model history
sbert_hist = np.load('../data/primary_model/training_history.npy', allow_pickle=True).item()

# Load SBERT test results from checkpoint
from dataset import ResumeDataset, split_dataset
from model import ResumeMatchNet
from train import _evaluate as sbert_evaluate

sbert_ds = ResumeDataset('../data/cleaned_resume_data.csv', cache_dir='../data/cache', fit=False)
_, _, sbert_test = split_dataset(sbert_ds, train=0.60, val=0.20, test=0.20, seed=42)
sbert_test_loader = DataLoader(sbert_test, batch_size=64, shuffle=False)

sbert_model = ResumeMatchNet(
    branch1_dim=sbert_ds.branch1_dim,
    branch2_dim=sbert_ds.branch2_dim,
).to(device)
ckpt = torch.load('../data/primary_model/best_model.pt', map_location=device, weights_only=False)
sbert_model.load_state_dict(ckpt['model_state'])

criterion = torch.nn.MSELoss()
s_rmse, s_mae, s_r2, s_acc, s_f1, _, _ = sbert_evaluate(sbert_model, sbert_test_loader, device, criterion)

# RF baseline (re-run on current data, separate 60/20/20 split)
rf_test_rmse = 0.1075
rf_test_r2   = 0.5821
rf_test_acc  = 0.8638
rf_test_f1   = 0.8531

# Comparison table
comparison = pd.DataFrame({
    'Model':     ['RF Baseline', 'SBERT + DNN', 'BiLSTM + DNN'],
    'Test RMSE': [rf_test_rmse, round(s_rmse, 4), round(results['test_rmse'], 4)],
    'Test R²':   [rf_test_r2, round(s_r2, 4), round(results['test_r2'], 4)],
    'Test Acc':  [rf_test_acc, round(s_acc, 4), round(results['test_acc'], 4)],
    'Test F1':   [rf_test_f1, round(s_f1, 4), round(results['test_f1'], 4)],
    'Params':    ['100 trees', f"{sbert_model.count_parameters():,}", f"{model.count_parameters():,}"],
})
print(comparison.to_string(index=False))

## 7. Qualitative Results — Sample Predictions

Show individual predictions to illustrate what the model does well and where it struggles.

In [ ]:
# Get RNN predictions on test set
train_idx, val_idx, test_idx = _split_indices(len(dataset), seed=42)
test_subset = Subset(dataset, test_idx)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False, collate_fn=rnn_collate_fn)

_, _, _, _, _, rnn_preds, rnn_targets = _evaluate(model, test_loader, device, criterion)

# Load raw data for context
df = pd.read_csv('../data/cleaned_resume_data.csv')
test_df = df.iloc[test_idx].copy()
test_df['predicted'] = rnn_preds
test_df['error'] = np.abs(rnn_preds - rnn_targets)

# Best predictions (lowest error)
print('=== BEST PREDICTIONS (lowest absolute error) ===')
best = test_df.nsmallest(5, 'error')
for _, row in best.iterrows():
    print(f"  True: {row['matched_score']:.3f}  Pred: {row['predicted']:.3f}  "
          f"Error: {row['error']:.4f}  Job: {row['job_position_name']}  "
          f"Degree: {row['degree_level']}")

print()
print('=== WORST PREDICTIONS (highest absolute error) ===')
worst = test_df.nlargest(5, 'error')
for _, row in worst.iterrows():
    print(f"  True: {row['matched_score']:.3f}  Pred: {row['predicted']:.3f}  "
          f"Error: {row['error']:.4f}  Job: {row['job_position_name']}  "
          f"Degree: {row['degree_level']}")

In [ ]:
# Error distribution and scatter plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residual histogram
residuals = rnn_preds - rnn_targets
ax = axes[0]
ax.hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=1)
ax.set_xlabel('Residual (pred - true)')
ax.set_ylabel('Count')
ax.set_title(f'Residual Distribution (mean={residuals.mean():.4f})')
ax.grid(True, alpha=0.3)

# Scatter: true vs predicted
ax = axes[1]
ax.scatter(rnn_targets, rnn_preds, alpha=0.2, s=10, color='teal')
lo, hi = min(rnn_targets.min(), rnn_preds.min()), max(rnn_targets.max(), rnn_preds.max())
ax.plot([lo, hi], [lo, hi], 'r--', lw=1, label='Perfect')
ax.set_xlabel('True matched_score')
ax.set_ylabel('Predicted matched_score')
ax.set_title('Predicted vs True (RNN Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# Error by degree level
ax = axes[2]
degree_errors = test_df.groupby('degree_level')['error'].agg(['mean', 'count'])
degree_errors = degree_errors.sort_values('mean')
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(degree_errors)))
bars = ax.barh(degree_errors.index, degree_errors['mean'], color=colors)
for bar, (_, row) in zip(bars, degree_errors.iterrows()):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'n={int(row["count"])}', va='center', fontsize=8)
ax.set_xlabel('Mean Absolute Error')
ax.set_title('Error by Degree Level')
ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('RNN Model — Qualitative Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Career Objective Subgroup Analysis

Compare RMSE for samples with vs without a career objective.

In [ ]:
has_obj = test_df['has_career_objective'] == 1

rmse_has  = np.sqrt(np.mean((rnn_preds[has_obj.values]  - rnn_targets[has_obj.values])**2))
rmse_no   = np.sqrt(np.mean((rnn_preds[~has_obj.values] - rnn_targets[~has_obj.values])**2))

print(f'Samples WITH career objective:    n={has_obj.sum():>4}  RMSE={rmse_has:.4f}')
print(f'Samples WITHOUT career objective: n={(~has_obj).sum():>4}  RMSE={rmse_no:.4f}')
print(f'Difference: {rmse_no - rmse_has:+.4f}')

## 9. Evaluate on New Data (Held-Out Test Set)

The test set (20% = 1,910 samples) was held out before any training and was
**never** used to influence hyperparameter tuning. The validation set was used
for early stopping; the test set is strictly for final evaluation.

Below we show per-decile performance and detailed per-sample predictions to
demonstrate the model's behaviour on truly unseen data.

In [ ]:
# Per-decile analysis: how accurate is the model across the score range?
test_df['score_bin'] = pd.cut(test_df['matched_score'], bins=10)
decile = test_df.groupby('score_bin').agg(
    count=('error', 'count'),
    mean_error=('error', 'mean'),
    rmse=('error', lambda x: np.sqrt(np.mean(x**2))),
    mean_true=('matched_score', 'mean'),
    mean_pred=('predicted', 'mean'),
).round(4)
print('=== Per-Decile Test Performance (RNN Model) ===')
print(decile.to_string())

In [ ]:
# Random sample of 10 test predictions for inspection
sample = test_df.sample(10, random_state=42)[[
    'job_position_name', 'degree_level', 'gpa_normalized',
    'total_work_experience_years', 'has_career_objective',
    'matched_score', 'predicted', 'error'
]].round(4)
sample.columns = ['Job', 'Degree', 'GPA', 'Work Exp (yrs)', 'Has Obj',
                   'True Score', 'Predicted', 'Abs Error']
print('=== Random Test Samples ===')
print(sample.to_string(index=False))

In [ ]:
# Bootstrap 95% confidence interval on test RMSE
n_bootstrap = 1000
rng = np.random.default_rng(42)
boot_rmse = []
for _ in range(n_bootstrap):
    idx = rng.choice(len(rnn_targets), size=len(rnn_targets), replace=True)
    boot_rmse.append(np.sqrt(np.mean((rnn_preds[idx] - rnn_targets[idx])**2)))
boot_rmse = np.array(boot_rmse)
ci_lo, ci_hi = np.percentile(boot_rmse, [2.5, 97.5])

print(f'Test RMSE:          {results["test_rmse"]:.4f}')
print(f'95% Bootstrap CI:   [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Test R²:            {results["test_r2"]:.4f}')
print(f'Test Accuracy:      {results["test_acc"]:.4f}  (threshold={MATCH_THRESHOLD})')
print(f'Test F1 (weighted): {results["test_f1"]:.4f}')